In [11]:
!pip install bokeh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 59.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 483.2/483.2 kB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 38.4 MB/s eta 0:00:00


In [31]:
import numpy as np
import networkx as nx
import gurobipy as gp
from gurobipy import GRB

# 1. GRAPH TOPOLOGY & NODE LAYOUT
G = nx.Graph()
G.add_nodes_from(range(1, 11))
edges = [
    (1, 5), 
    (2, 5),
    (3, 4),
    (4, 5), (4, 7),
    (5, 6),
    (7, 8), (7, 9),
    (8, 9), (8, 10),
    (9, 10)
]
G.add_edges_from(edges)

node_positions = {
    1: (7.0, 7.0),
    2: (7.5, 5.0),
    3: (8.5, 3.0),
    4: (5.5, 3.0),
    5: (5.0, 5.5),
    6: (4.5, 7.5),
    7: (3.5, 3.0),
    8: (3.0, 5.5),
    9: (1.5, 3.0),
    10: (2.5, 7.5)
}

directed_edges = list(G.edges()) + [(v, u) for u, v in G.edges()]
edge_xs = [[node_positions[u][0], node_positions[v][0]] for u, v in G.edges()]
edge_ys = [[node_positions[u][1], node_positions[v][1]] for u, v in G.edges()]

def run_simulation(
    rounds=20,
    budget=1000.0,
    c_trans=0.10,
    alpha_open=1.0,
    alpha_covert=0.25,
    penalty_open=0.85,
    penalty_covert=0.10,
    penalty_idle=0.60,
    v_material=2.5,
    facility_cap=20.0,
    cooling_rate=0.55,
    breakout_goal=2000.0,
    seed=42
):
    np.random.seed(int(seed))
    blocks = np.random.choice(10, size=20)
    m_init = np.bincount(blocks, minlength=10) * 10.0
    h_init = np.zeros(10)

    history = [{
        'round': 0,
        'm': m_init.copy(),
        'h': h_init.copy(),
        'n_open': np.zeros(10),
        'n_covert': np.zeros(10),
        's_idle': m_init.copy(),
        'losses': np.zeros(10),
        'inspections': np.zeros(10),
        'node_progress': np.zeros(10),
        'flows': {},
        'progress': 0.0,
        'cum_progress': 0.0,
        'cum_seized': 0.0,
        'inflows': np.zeros(10),
        'outflows': np.zeros(10)
    }]

    curr_m = m_init.copy()
    curr_h = h_init.copy()
    cum_p = 0.0
    cum_s = 0.0

    for t in range(1, rounds + 1):
        total_h = np.sum(curr_h)
        if total_h > 1e-3:
            pi_est = np.clip(3.0 * (curr_h / total_h), 0.05, 0.95)
        else:
            pi_est = np.full(10, 0.30)

        random_tie = np.random.uniform(0.0, 1e-4, size=10)
        top_nodes = np.argsort(curr_h + random_tie)[::-1][:3]
        actual_insp = np.zeros(10)
        actual_insp[top_nodes] = 1.0

        model = gp.Model(f"Round_{t}")
        model.setParam('OutputFlag', 0)

        n_open = model.addVars(10, lb=0.0, name="n_open")
        n_covert = model.addVars(10, lb=0.0, name="n_covert")
        s_idle = model.addVars(10, lb=0.0, name="s_idle")
        flow_vars = model.addVars(directed_edges, lb=0.0, name="flows")
        m_next = model.addVars(10, lb=0.0, name="m_next")

        obj = (
            gp.quicksum((1.0 - pi_est[i]) * alpha_open * n_open[i] + alpha_covert * n_covert[i] for i in range(10))
            + v_material * gp.quicksum(m_next[i] for i in range(10))
            - gp.quicksum(c_trans * flow_vars[e] for e in directed_edges)
        )
        model.setObjective(obj, GRB.MAXIMIZE)

        for i in range(10):
            node_id = i + 1
            outflow = gp.quicksum(flow_vars[(node_id, j)] for j in G.neighbors(node_id))
            model.addConstr(n_open[i] + n_covert[i] + s_idle[i] + outflow == curr_m[i])
            model.addConstr(n_open[i] + n_covert[i] <= facility_cap)

        for i in range(10):
            node_id = i + 1
            inflow = gp.quicksum(flow_vars[(j, node_id)] for j in G.neighbors(node_id))
            surv_open = (1.0 - pi_est[i] * penalty_open) * n_open[i]
            surv_covert = (1.0 - pi_est[i] * penalty_covert) * n_covert[i]
            surv_idle = (1.0 - pi_est[i] * penalty_idle) * s_idle[i]
            surv_inflow = (1.0 - pi_est[i] * penalty_idle) * inflow
            model.addConstr(m_next[i] == surv_open + surv_covert + surv_idle + surv_inflow)

        model.addConstr(
            gp.quicksum(n_open[i] + n_covert[i] for i in range(10))
            + gp.quicksum(c_trans * flow_vars[e] for e in directed_edges) <= budget
        )

        model.optimize()

        sol_open = np.array([n_open[i].X for i in range(10)])
        sol_covert = np.array([n_covert[i].X for i in range(10)])
        sol_idle = np.array([s_idle[i].X for i in range(10)])
        flows = {e: flow_vars[e].X for e in directed_edges if flow_vars[e].X > 0.1}

        inflow_actual = np.zeros(10)
        outflow_actual = np.zeros(10)
        for (u, v), f_val in flows.items():
            outflow_actual[u - 1] += f_val
            inflow_actual[v - 1] += f_val

        node_progress = (1.0 - actual_insp) * alpha_open * sol_open + alpha_covert * sol_covert
        round_progress = np.sum(node_progress)

        node_losses = actual_insp * (
            penalty_open * sol_open +
            penalty_covert * sol_covert +
            penalty_idle * sol_idle +
            penalty_idle * inflow_actual
        )

        cum_p += round_progress
        cum_s += np.sum(node_losses)

        history.append({
            'round': t,
            'm': curr_m.copy(),
            'h': curr_h.copy(),
            'n_open': sol_open.copy(),
            'n_covert': sol_covert.copy(),
            's_idle': sol_idle.copy(),
            'losses': node_losses.copy(),
            'inspections': actual_insp.copy(),
            'node_progress': node_progress.copy(),
            'flows': flows.copy(),
            'progress': round_progress,
            'cum_progress': cum_p,
            'cum_seized': cum_s,
            'inflows': inflow_actual.copy(),
            'outflows': outflow_actual.copy()
        })

        curr_m = np.maximum(0.0, curr_m - outflow_actual + inflow_actual - node_losses)
        curr_h = cooling_rate * curr_h + (1.0 * sol_open + 0.15 * sol_covert)

    return history, breakout_goal

In [32]:
import numpy as np
import networkx as nx
import gurobipy as gp
from gurobipy import GRB

from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, Slider, Button, CustomJS, Arrow, NormalHead, Div
from bokeh.layouts import column, row

import ipywidgets as widgets
from IPython.display import display, clear_output

output_notebook()

# ---------------------------------------------------------
# 1. NETWORK TOPOLOGY & NODE LAYOUT
# ---------------------------------------------------------
G = nx.Graph()
G.add_nodes_from(range(1, 11))
edges = [
    (1, 5),
    (2, 5),
    (3, 4),
    (4, 5), (4, 7),
    (5, 6),
    (7, 8), (7, 9),
    (8, 9), (8, 10),
    (9, 10)
]
G.add_edges_from(edges)

node_positions = {
    1: (7.0, 7.0),
    2: (7.5, 5.0),
    3: (8.5, 3.0),
    4: (5.5, 3.0),
    5: (5.0, 5.5),
    6: (4.5, 7.5),
    7: (3.5, 3.0),
    8: (3.0, 5.5),
    9: (1.5, 3.0),
    10: (2.5, 7.5)
}

directed_edges = list(G.edges()) + [(v, u) for u, v in G.edges()]
edge_xs = [[node_positions[u][0], node_positions[v][0]] for u, v in G.edges()]
edge_ys = [[node_positions[u][1], node_positions[v][1]] for u, v in G.edges()]

# ---------------------------------------------------------
# 2. SIMULATION ENGINE (GUROBI OPTIMIZER)
# ---------------------------------------------------------
def run_simulation(
    rounds=20,
    budget=1000.0,
    import_amount=10.0,
    c_trans=0.10,
    alpha_open=1.0,
    alpha_covert=0.25,
    penalty_open=0.85,
    penalty_covert=0.10,
    penalty_idle=0.60,
    v_material=2.5,
    facility_cap=20.0,
    cooling_rate=0.55,
    breakout_goal=2000.0,
    seed=42
):
    np.random.seed(int(seed))
    blocks = np.random.choice(10, size=20)
    m_init = np.bincount(blocks, minlength=10) * 10.0
    h_init = np.zeros(10)

    history = [{
        'round': 0,
        'm': m_init.copy(),
        'h': h_init.copy(),
        'n_open': np.zeros(10),
        'n_covert': np.zeros(10),
        's_idle': m_init.copy(),
        'imports': np.zeros(10),
        'losses': np.zeros(10),
        'inspections': np.zeros(10),
        'node_progress': np.zeros(10),
        'flows': {},
        'progress': 0.0,
        'cum_progress': 0.0,
        'cum_seized': 0.0,
        'inflows': np.zeros(10),
        'outflows': np.zeros(10)
    }]

    curr_m = m_init.copy()
    curr_h = h_init.copy()
    cum_p = 0.0
    cum_s = 0.0

    for t in range(1, rounds + 1):
        # Country B inspection forecast based purely on thermal signature
        total_h = np.sum(curr_h)
        if total_h > 1e-3:
            pi_est = np.clip(3.0 * (curr_h / total_h), 0.05, 0.95)
        else:
            pi_est = np.full(10, 0.30)

        # Country A target choice (strictly heat + random tiebreaker, blind to stockpiles)
        random_tie = np.random.uniform(0.0, 1e-4, size=10)
        top_nodes = np.argsort(curr_h + random_tie)[::-1][:3]
        actual_insp = np.zeros(10)
        actual_insp[top_nodes] = 1.0

        model = gp.Model(f"Round_{t}")
        model.setParam('OutputFlag', 0)

        # Variables
        n_open = model.addVars(10, lb=0.0, name="n_open")
        n_covert = model.addVars(10, lb=0.0, name="n_covert")
        s_idle = model.addVars(10, lb=0.0, name="s_idle")
        flow_vars = model.addVars(directed_edges, lb=0.0, name="flows")
        m_next = model.addVars(10, lb=0.0, name="m_next")
        
        # Imports decided in round t to arrive in m_{i, t+1} across nodes 1, 2, 3
        imports = model.addVars(3, lb=0.0, name="imports")
        model.addConstr(imports[0] + imports[1] + imports[2] == import_amount)

        # Objective function
        obj = (
            gp.quicksum((1.0 - pi_est[i]) * alpha_open * n_open[i] + alpha_covert * n_covert[i] for i in range(10))
            + v_material * gp.quicksum(m_next[i] for i in range(10))
            - gp.quicksum(c_trans * flow_vars[e] for e in directed_edges)
        )
        model.setObjective(obj, GRB.MAXIMIZE)

        # 1. Round t material availability: only current stockpile can be processed or transported
        for i in range(10):
            node_id = i + 1
            outflow = gp.quicksum(flow_vars[(node_id, j)] for j in G.neighbors(node_id))
            model.addConstr(n_open[i] + n_covert[i] + s_idle[i] + outflow == curr_m[i])
            model.addConstr(n_open[i] + n_covert[i] <= facility_cap)

        # 2. Inventory transition: imports arrive directly into m_{i, t+1}
        for i in range(10):
            node_id = i + 1
            inflow = gp.quicksum(flow_vars[(j, node_id)] for j in G.neighbors(node_id))
            surv_open = (1.0 - pi_est[i] * penalty_open) * n_open[i]
            surv_covert = (1.0 - pi_est[i] * penalty_covert) * n_covert[i]
            surv_idle = (1.0 - pi_est[i] * penalty_idle) * s_idle[i]
            surv_inflow = (1.0 - pi_est[i] * penalty_idle) * inflow
            
            # Add imports to next turn's stockpile for nodes 1-3
            in_import = imports[i] if i < 3 else 0.0
            model.addConstr(m_next[i] == surv_open + surv_covert + surv_idle + surv_inflow + in_import)

        # Budget constraint
        model.addConstr(
            gp.quicksum(n_open[i] + n_covert[i] for i in range(10))
            + gp.quicksum(c_trans * flow_vars[e] for e in directed_edges) <= budget
        )

        model.optimize()

        sol_open = np.array([n_open[i].X for i in range(10)])
        sol_covert = np.array([n_covert[i].X for i in range(10)])
        sol_idle = np.array([s_idle[i].X for i in range(10)])
        flows = {e: flow_vars[e].X for e in directed_edges if flow_vars[e].X > 0.1}

        sol_imports = np.zeros(10)
        sol_imports[0] = imports[0].X
        sol_imports[1] = imports[1].X
        sol_imports[2] = imports[2].X

        inflow_actual = np.zeros(10)
        outflow_actual = np.zeros(10)
        for (u, v), f_val in flows.items():
            outflow_actual[u - 1] += f_val
            inflow_actual[v - 1] += f_val

        # Progress calculation: intercepted open research is destroyed
        node_progress = (1.0 - actual_insp) * alpha_open * sol_open + alpha_covert * sol_covert
        round_progress = np.sum(node_progress)

        # Seizure losses across research and idle stockpiles at inspected nodes
        node_losses = actual_insp * (
            penalty_open * sol_open +
            penalty_covert * sol_covert +
            penalty_idle * sol_idle +
            penalty_idle * inflow_actual
        )

        cum_p += round_progress
        cum_s += np.sum(node_losses)

        history.append({
            'round': t,
            'm': curr_m.copy(),
            'h': curr_h.copy(),
            'imports': sol_imports.copy(),
            'n_open': sol_open.copy(),
            'n_covert': sol_covert.copy(),
            's_idle': sol_idle.copy(),
            'losses': node_losses.copy(),
            'inspections': actual_insp.copy(),
            'node_progress': node_progress.copy(),
            'flows': flows.copy(),
            'progress': round_progress,
            'cum_progress': cum_p,
            'cum_seized': cum_s,
            'inflows': inflow_actual.copy(),
            'outflows': outflow_actual.copy()
        })

        # Next turn stockpile receives surviving stock + arriving imports
        curr_m = np.maximum(0.0, curr_m - outflow_actual + inflow_actual - node_losses) + sol_imports
        curr_h = cooling_rate * curr_h + (1.0 * sol_open + 0.15 * sol_covert)

    return history, breakout_goal

# ---------------------------------------------------------
# 3. DASHBOARD RENDERER (BOKEH HTML BUILDER)
# ---------------------------------------------------------
def render_bokeh_dashboard(history, breakout_goal):
    all_node_data, all_arrow_data, all_round_info = [], [], []
    all_kpi_html, all_preview_html, all_verbose_html = [], [], []

    # Radial offset in plot coordinates to stop arrows cleanly at circle edges
    NODE_RAD = 0.38

    for idx, state in enumerate(history):
        r_num = state['round']
        cum_progress = state['cum_progress']
        cum_seized = state['cum_seized']
        pct_breakout = min(100.0, (cum_progress / breakout_goal) * 100.0)

        # Scorecard KPI Banner
        kpi_html = f"""
        <div style="display: flex; gap: 14px; font-family: -apple-system, BlinkMacSystemFont, sans-serif; margin-bottom: 6px;">
            <div style="flex: 1; padding: 10px 14px; background: #f8fafc; border: 1px solid #e2e8f0; border-radius: 8px;">
                <div style="font-size: 11px; text-transform: uppercase; color: #64748b; font-weight: 600;">Total Research Progress</div>
                <div style="font-size: 22px; font-weight: 700; color: #0284c7;">{cum_progress:,.1f}</div>
                <div style="font-size: 11px; color: #0369a1;">+{state['progress']:.1f} this round</div>
            </div>
            <div style="flex: 1; padding: 10px 14px; background: #f8fafc; border: 1px solid #e2e8f0; border-radius: 8px;">
                <div style="font-size: 11px; text-transform: uppercase; color: #64748b; font-weight: 600;">Total Seized Material</div>
                <div style="font-size: 22px; font-weight: 700; color: #dc2626;">{cum_seized:,.1f}</div>
                <div style="font-size: 11px; color: #b91c1c;">+{np.sum(state['losses']):.1f} seized this round</div>
            </div>
            <div style="flex: 1.2; padding: 10px 14px; background: #f8fafc; border: 1px solid #e2e8f0; border-radius: 8px;">
                <div style="font-size: 11px; text-transform: uppercase; color: #64748b; font-weight: 600;">Breakout Readiness ({breakout_goal:,.0f} Goal)</div>
                <div style="font-size: 22px; font-weight: 700; color: #16a34a;">{pct_breakout:.1f}%</div>
                <div style="width: 100%; background: #e2e8f0; border-radius: 4px; height: 6px; margin-top: 6px; overflow: hidden;">
                    <div style="background: #16a34a; width: {pct_breakout}%; height: 6px;"></div>
                </div>
            </div>
        </div>
        """
        all_kpi_html.append(kpi_html)

        # Forecast Panel for Round t+1
        if idx + 1 < len(history):
            next_state = history[idx + 1]
            next_inspected = [i for i in range(1, 11) if next_state['inspections'][i - 1] > 0]
            insp_badges = " ".join([f"<span style='background:#fee2e2; color:#b91c1c; padding:2px 7px; border-radius:4px; font-weight:600; font-size:11px;'>Node {n}</span>" for n in next_inspected])
            open_list = [f"N{i}({next_state['n_open'][i-1]:.0f})" for i in range(1, 11) if next_state['n_open'][i - 1] > 0.1]
            covert_list = [f"N{i}({next_state['n_covert'][i-1]:.0f})" for i in range(1, 11) if next_state['n_covert'][i - 1] > 0.1]
            flow_list = [f"{u}→{v}({amt:.0f})" for (u, v), amt in next_state['flows'].items()]
            import_list = [f"N{i+1}: +{next_state['imports'][i]:.1f}" for i in range(3) if next_state['imports'][i] > 0.05]
            
            preview_html = f"""
            <div style="display: flex; gap: 12px; font-family: -apple-system, BlinkMacSystemFont, sans-serif; margin-top: 8px;">
                <div style="flex: 1; padding: 10px 12px; background: #fff1f2; border: 1px solid #fecdd3; border-radius: 8px;">
                    <div style="font-size: 11px; font-weight: 700; color: #9f1239; text-transform: uppercase;">🎯 Country A Planned Inspections (Round {next_state['round']})</div>
                    <div style="display: flex; gap: 6px; margin-top: 5px;">{insp_badges}</div>
                </div>
                <div style="flex: 1.4; padding: 10px 12px; background: #f0fdf4; border: 1px solid #bbf7d0; border-radius: 8px;">
                    <div style="font-size: 11px; font-weight: 700; color: #166534; text-transform: uppercase;">🔬 Country B Planned Actions (Round {next_state['round']})</div>
                    <div style="font-size: 11px; color: #1e293b; margin-top: 3px; line-height: 1.4;">
                        <b>Incoming Imports (Landing in N1-3):</b> {', '.join(import_list) if import_list else 'None'}<br>
                        <b>Open Research:</b> {', '.join(open_list) if open_list else 'None'} | <b>Covert:</b> {', '.join(covert_list) if covert_list else 'None'}<br>
                        <b>Transports:</b> {', '.join(flow_list) if flow_list else 'None'}
                    </div>
                </div>
            </div>
            """
        else:
            preview_html = "<div style='padding: 10px; background: #f1f5f9; border-radius: 8px; text-align: center; color: #64748b; font-weight: 600; font-size: 12px; margin-top: 8px;'>🏁 Simulation Complete</div>"
        all_preview_html.append(preview_html)

        # Verbose per-Node Breakdown Table
        rows_html = ""
        for i in range(1, 11):
            is_insp = state['inspections'][i - 1] > 0
            tag_insp = "<span style='color: #dc2626; font-weight: bold;'>YES</span>" if is_insp else "<span style='color: #64748b;'>No</span>"
            o_amt = state['n_open'][i - 1]
            c_amt = state['n_covert'][i - 1]
            m_start = state['m'][i - 1]
            imp_amt = state['imports'][i - 1]
            seized = state['losses'][i - 1]
            prog = state['node_progress'][i - 1]
            net_f = state['inflows'][i - 1] - state['outflows'][i - 1]
            prog_calc = f"{prog:.1f} (Intercepted!)" if (is_insp and o_amt > 0.1) else f"{prog:.1f}"
            bg_row = "#fff1f2" if is_insp else ("#f8fafc" if i % 2 == 0 else "#ffffff")

            imp_display = f"<span style='color:#059669; font-weight:600;'>+{imp_amt:.1f}</span>" if imp_amt > 0.05 else "-"

            rows_html += f"""
            <tr style="background: {bg_row}; border-bottom: 1px solid #e2e8f0;">
                <td style="padding: 6px 8px; font-weight: 600;">Node {i}</td>
                <td style="padding: 6px 8px; text-align: right;">{m_start:.1f}</td>
                <td style="padding: 6px 8px; text-align: center;">{imp_display}</td>
                <td style="padding: 6px 8px; text-align: right;">{o_amt:.1f}</td>
                <td style="padding: 6px 8px; text-align: right;">{c_amt:.1f}</td>
                <td style="padding: 6px 8px; text-align: right;">{'+' if net_f > 0 else ''}{net_f:.1f}</td>
                <td style="padding: 6px 8px; text-align: center;">{tag_insp}</td>
                <td style="padding: 6px 8px; text-align: right; color: #dc2626; font-weight: 600;">{seized:.1f}</td>
                <td style="padding: 6px 8px; text-align: right; color: #0284c7; font-weight: 600;">{prog_calc}</td>
            </tr>
            """

        verbose_html = f"""
        <div style="margin-top: 10px; font-family: -apple-system, BlinkMacSystemFont, sans-serif; font-size: 11px;">
            <div style="font-weight: 700; color: #334155; margin-bottom: 6px; font-size: 12px; text-transform: uppercase;">
                📋 Verbose Round {r_num} per-Node Math Breakdown
            </div>
            <div style="max-height: 220px; overflow-y: auto; border: 1px solid #cbd5e1; border-radius: 6px;">
                <table style="width: 100%; border-collapse: collapse; text-align: left;">
                    <thead>
                        <tr style="background: #e2e8f0; color: #475569; position: sticky; top: 0;">
                            <th style="padding: 6px 8px;">Facility</th>
                            <th style="padding: 6px 8px; text-align: right;">Start Mat</th>
                            <th style="padding: 6px 8px; text-align: center;">Added Import</th>
                            <th style="padding: 6px 8px; text-align: right;">Open Res</th>
                            <th style="padding: 6px 8px; text-align: right;">Covert Res</th>
                            <th style="padding: 6px 8px; text-align: right;">Net Transport</th>
                            <th style="padding: 6px 8px; text-align: center;">Inspected?</th>
                            <th style="padding: 6px 8px; text-align: right;">Materiel Seized</th>
                            <th style="padding: 6px 8px; text-align: right;">Progress Added</th>
                        </tr>
                    </thead>
                    <tbody>{rows_html}</tbody>
                </table>
            </div>
        </div>
        """
        all_verbose_html.append(verbose_html)

        # Node labels and circle fills
        xs = [node_positions[i][0] for i in range(1, 11)]
        ys = [node_positions[i][1] for i in range(1, 11)]
        labels, colors = [], []
        for i in range(1, 11):
            m_val = state['m'][i - 1]
            h_val = state['h'][i - 1]
            labels.append(f"Node {i}\nMat: {m_val:.0f}\nHeat: {h_val:.1f}")
            if state['inspections'][i - 1] > 0:
                colors.append("#ff5252")
            else:
                norm_h = min(1.0, h_val / 50.0)
                rb = int((1.0 - norm_h) * 255)
                colors.append(f"#{rb:02x}d9{rb:02x}")
        all_node_data.append({'x': xs, 'y': ys, 'colors': colors, 'labels': labels})

        # Offsetting transport arrows to touch node circle perimeters
        ax_start, ay_start, ax_end, ay_end = [], [], [], []
        for (u, v), flow in state['flows'].items():
            if flow > 0.1:
                x1, y1 = node_positions[u]
                x2, y2 = node_positions[v]
                dx = x2 - x1
                dy = y2 - y1
                dist = np.hypot(dx, dy)
                if dist > 1e-4:
                    ux = dx / dist
                    uy = dy / dist
                    ax_start.append(x1 + ux * NODE_RAD)
                    ay_start.append(y1 + uy * NODE_RAD)
                    ax_end.append(x2 - ux * NODE_RAD)
                    ay_end.append(y2 - uy * NODE_RAD)
        all_arrow_data.append({'x_start': ax_start, 'y_start': ay_start, 'x_end': ax_end, 'y_end': ay_end})

        if r_num == 0:
            all_round_info.append("Round 0 / 20  |  Initial Setup (200 Units Distributed)")
        else:
            inspected = [i for i in range(1, 11) if state['inspections'][i - 1] > 0]
            all_round_info.append(f"Round {r_num} / 20  |  Country A Inspected: {inspected}")

    # Build Bokeh Figure
    source_nodes = ColumnDataSource(data=all_node_data[0])
    source_arrows = ColumnDataSource(data=all_arrow_data[0])
    source_info = ColumnDataSource(data=dict(
        round_info=all_round_info,
        kpi_html=all_kpi_html,
        preview_html=all_preview_html,
        verbose_html=all_verbose_html,
        all_nodes=all_node_data,
        all_arrows=all_arrow_data
    ))

    kpi_div = Div(text=all_kpi_html[0], width=800)
    preview_div = Div(text=all_preview_html[0], width=800)
    verbose_div = Div(text=all_verbose_html[0], width=800)

    p = figure(
        title=all_round_info[0],
        width=800, height=500,
        x_range=(0.5, 9.5), y_range=(2.0, 8.5),
        tools=""
    )
    p.axis.visible = False
    p.grid.visible = False
    p.title.text_font_size = "11pt"

    p.multi_line(xs=edge_xs, ys=edge_ys, color="#888888", line_width=2.5)

    arrow_head = NormalHead(fill_color="#2196F3", line_color="#1976D2", size=13)
    p.add_layout(Arrow(
        end=arrow_head,
        x_start='x_start', y_start='y_start',
        x_end='x_end', y_end='y_end',
        source=source_arrows,
        line_color="#2196F3",
        line_width=3
    ))

    # Larger node circle size: 58
    p.scatter('x', 'y', size=58, fill_color='colors', line_color='black', line_width=2.0, source=source_nodes)
    p.text('x', 'y', text='labels', source=source_nodes, text_align='center', text_baseline='middle', text_font_size='8.5pt')

    slider = Slider(start=0, end=20, value=0, step=1, title="Round", width=340)
    btn_prev = Button(label="◀ Prev", width=70, button_type="default")
    btn_play = Button(label="Play ▶", width=80, button_type="primary")
    btn_next = Button(label="Next ▶", width=70, button_type="default")
    btn_reset = Button(label="↺ Reset to t=0", width=110, button_type="warning")

    # Client-side JavaScript updates
    update_view_js = CustomJS(args=dict(
        slider=slider, s_nodes=source_nodes, s_arrows=source_arrows,
        s_info=source_info, plot=p, kpi_div=kpi_div,
        preview_div=preview_div, verbose_div=verbose_div
    ), code="""
        const r_idx = slider.value;
        const all_nodes = s_info.data['all_nodes'];
        const all_arrows = s_info.data['all_arrows'];
        const round_info = s_info.data['round_info'];
        const kpi_html = s_info.data['kpi_html'];
        const preview_html = s_info.data['preview_html'];
        const verbose_html = s_info.data['verbose_html'];

        kpi_div.text = kpi_html[r_idx];
        preview_div.text = preview_html[r_idx];
        verbose_div.text = verbose_html[r_idx];
        plot.title.text = round_info[r_idx];
        
        s_nodes.data = all_nodes[r_idx];
        s_nodes.change.emit();
        s_arrows.data = all_arrows[r_idx];
        s_arrows.change.emit();
    """)

    slider.js_on_change('value', update_view_js)
    btn_prev.js_on_click(CustomJS(args=dict(slider=slider), code="if(slider.value > slider.start){ slider.value -= 1; }"))
    btn_next.js_on_click(CustomJS(args=dict(slider=slider), code="if(slider.value < slider.end){ slider.value += 1; }"))
    btn_reset.js_on_click(CustomJS(args=dict(slider=slider), code="slider.value = 0;"))

    btn_play.js_on_click(CustomJS(args=dict(slider=slider, btn=btn_play), code="""
        if (window.bokeh_game_timer) {
            clearInterval(window.bokeh_game_timer);
            window.bokeh_game_timer = null;
            btn.label = "Play ▶";
        } else {
            btn.label = "Pause ⏸";
            window.bokeh_game_timer = setInterval(function() {
                if (slider.value < slider.end) {
                    slider.value += 1;
                } else {
                    clearInterval(window.bokeh_game_timer);
                    window.bokeh_game_timer = null;
                    btn.label = "Play ▶";
                }
            }, 900);
        }
    """))

    controls = row(btn_prev, btn_play, btn_next, btn_reset, slider)
    return column(kpi_div, p, controls, preview_div, verbose_div)

# ---------------------------------------------------------
# 4. IPYWIDGETS HYPERPARAMETER CONTROL INTERFACE
# ---------------------------------------------------------
w_vmat = widgets.FloatSlider(value=2.5, min=0.5, max=5.0, step=0.1, description='V_mat:', readout_format='.1f')
w_fcap = widgets.FloatSlider(value=20.0, min=5.0, max=50.0, step=5.0, description='Facility Cap:', readout_format='.0f')
w_ctrans = widgets.FloatSlider(value=0.10, min=0.01, max=1.0, step=0.01, description='Trans Cost:', readout_format='.2f')
w_import = widgets.FloatSlider(value=10.0, min=0.0, max=30.0, step=5.0, description='Imports/Rnd:', readout_format='.0f')

w_a_open = widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description='Open Speed:', readout_format='.1f')
w_a_covert = widgets.FloatSlider(value=0.25, min=0.05, max=1.0, step=0.05, description='Covert Speed:', readout_format='.2f')
w_pen_open = widgets.FloatSlider(value=0.85, min=0.2, max=1.0, step=0.05, description='Open Pen:', readout_format='.2f')
w_pen_idle = widgets.FloatSlider(value=0.60, min=0.1, max=1.0, step=0.05, description='Idle Pen:', readout_format='.2f')

w_cooling = widgets.FloatSlider(value=0.55, min=0.1, max=0.9, step=0.05, description='Cooling (λ):', readout_format='.2f')
w_seed = widgets.IntText(value=42, description='Seed:')
btn_apply = widgets.Button(description='⚡ Apply Settings & Run', button_style='success', icon='play', layout=widgets.Layout(width='220px'))

dash_output = widgets.Output()

def execute_and_display(b=None):
    with dash_output:
        clear_output(wait=True)
        # 1. Run optimization model with current widget values
        hist, goal = run_simulation(
            v_material=w_vmat.value,
            facility_cap=w_fcap.value,
            c_trans=w_ctrans.value,
            import_amount=w_import.value,
            alpha_open=w_a_open.value,
            alpha_covert=w_a_covert.value,
            penalty_open=w_pen_open.value,
            penalty_idle=w_pen_idle.value,
            cooling_rate=w_cooling.value,
            seed=w_seed.value
        )
        # 2. Render Bokeh dashboard (automatically initializes at Round 0)
        dashboard_layout = render_bokeh_dashboard(hist, goal)
        show(dashboard_layout)

btn_apply.on_click(execute_and_display)

# Layout Assembly
col1 = widgets.VBox([w_vmat, w_fcap, w_ctrans, w_import])
col2 = widgets.VBox([w_a_open, w_a_covert, w_pen_open, w_pen_idle])
col3 = widgets.VBox([w_cooling, w_seed, btn_apply])

panel_box = widgets.VBox([
    widgets.HTML("<h3 style='margin: 0 0 10px 0;'>⚙️ Simulation Hyperparameter Controls</h3>"),
    widgets.HBox([col1, col2, col3], layout=widgets.Layout(border='1px solid #ddd', padding='10px', border_radius='8px'))
])

display(panel_box, dash_output)
execute_and_display()